In [1]:
import csv

In [2]:
from time import strptime

In [3]:
import pandas as pd

In [4]:
from geojson import Feature, FeatureCollection, Point

In [5]:
import json

In [11]:
import numpy as np

In [7]:
filepath = './xls_datasets/map_3.xlsx'

In [8]:
workbook = pd.ExcelFile(filepath)
sheets = workbook.sheet_names

df = pd.concat([pd.read_excel(workbook, sheet_name=s)
                .assign(sheet_name=s) for s in sheets], ignore_index=True)

In [9]:
df.tail(5)

,coordinates,day,month,sheet_name,type,scale,description
415,"46.959077, 32.011810",18,July,Цивільна інфраструктура,point,NaN,У Миколаєві ракети влучили в автосалон та сало...
416,"47.041280, 32.438662",22,July,Цивільна інфраструктура,point,NaN,Влучання в будівлю магазину та гаражний коопер...
417,"46.959077, 32.011812",28,July,Цивільна інфраструктура,point,NaN,Ракетний удар по Миколаєву пошкодив яхт-клуб.
418,"46.959077, 32.011812",28,July,Цивільна інфраструктура,point,NaN,У Миколаєві пошкоджено пункт видачі гуманітарн...
419,"46.959077, 32.011812",31,July,Цивільна інфраструктура,point,NaN,Внаслідок масованих обстрілів Миколаєва пошкод...


In [10]:
df = df.dropna(subset=['day']) # dropping all Null values in column

In [11]:
df.loc[df['scale'].notnull()]

,coordinates,day,month,sheet_name,type,scale,description
227,"47.090263, 32.448873",1,July,Загиблі цивільні,NaN,1.0,"В с. Білозірка пошкоджено будинок, загинула од..."
228,"46.826064, 32.219268",2,July,Загиблі цивільні,NaN,1.0,В селищі Луч внаслідок ворожих обстрілів загин...
229,"47.023995, 32.427847",5,July,Загиблі цивільні,NaN,1.0,Внаслідок обстрілів с. Засілля загинула одна л...
230,"46.942736, 31.552642",6,July,Загиблі цивільні,NaN,1.0,Під час обстрілів с. Нечаяне загинула одна люд...
231,"46.824849, 32.141298",7,July,Загиблі цивільні,NaN,2.0,Внаслідок обстрілів с. Українка Галицинівської...
232,"47.403720, 32.438221",12,July,Загиблі цивільні,NaN,1.0,Через ракетний удар по Баштанці загинула жінка.
233,"46.999938, 32.395752",13,July,Загиблі цивільні,NaN,4.0,Через обстріли с. Новоселівка загинуло 4 людини.
234,"47.403720, 32.438221",14,July,Загиблі цивільні,NaN,1.0,Через нічний обстріл Баштанки загинула одна лю...
235,"46.862150, 32.196443",16,July,Загиблі цивільні,NaN,3.0,У с. Шевченкове внаслідок обстрілів загинули т...
236,"47.123866, 32.621648",16,July,Загиблі цивільні,NaN,1.0,Через обстріли у Широківській громаді загинула...


In [12]:
df

,coordinates,day,month,sheet_name,type,scale,description
0,"46.959632, 32.006337",1,July,Ракети,NaN,NaN,NaN
1,"46.959077, 32.011807",1,July,Ракети,NaN,NaN,NaN
2,"46.959077, 32.011807",5,July,Ракети,NaN,NaN,NaN
3,"46.959077, 32.011807",11,July,Ракети,NaN,NaN,NaN
4,"46.959077, 32.011807",12,July,Ракети,NaN,NaN,NaN
...,...,...,...,...,...,...,...
415,"46.959077, 32.011810",18,July,Цивільна інфраструктура,point,NaN,У Миколаєві ракети влучили в автосалон та сало...
416,"47.041280, 32.438662",22,July,Цивільна інфраструктура,point,NaN,Влучання в будівлю магазину та гаражний коопер...
417,"46.959077, 32.011812",28,July,Цивільна інфраструктура,point,NaN,Ракетний удар по Миколаєву пошкодив яхт-клуб.
418,"46.959077, 32.011812",28,July,Цивільна інфраструктура,point,NaN,У Миколаєві пошкоджено пункт видачі гуманітарн...


In [13]:
df[['la', 'lo']] = df['coordinates'].str.split(',', 1, expand=True) #split column coordinates into two

/tmp/ipykernel_44754/1141815606.py:1: FutureWarning: In a future version of pandas all arguments of StringMethods.split except for the argument 'pat' will be keyword-only.
  df[['la', 'lo']] = df['coordinates'].str.split(',', 1, expand=True) #split column coordinates into two


In [14]:
del(df['coordinates']) #deleting coordinates column

In [15]:
#del(df['type']) #deleting type column

In [16]:
df['la'] = df['la'].astype(float) #assigning float to la and lo column

In [17]:
df['lo'] = df['lo'].astype(float)

In [18]:
df['day'] = df['day'].astype(int)

In [19]:
# froming date and unix timestamp columns
def date_convert(df):
    for i in range(len(df)):
    #for i in range(10):
        month = df['month'][1]
        s = month.strip()[:3] #strip to 3 letter
        m = strptime(s,'%b').tm_mon #getting month number
        #print(m)
        time = (f"2022-{m}-{df['day'][i]}")
        #print(time)
        df.loc[i,['date']] = time
        time_ms = pd.Timestamp(time).timestamp()
        df.loc[i,['timestamp']] = time_ms*1000

In [20]:
date_convert(df)

In [21]:
df.loc[50:65]

,day,month,sheet_name,type,scale,description,la,lo,date,timestamp
50,5,July,Артилерія та РСЗВ,NaN,NaN,NaN,47.198526,32.829574,2022-7-5,1.656979e+12
51,4,July,Артилерія та РСЗВ,NaN,NaN,NaN,46.727960,31.972350,2022-7-4,1.656893e+12
52,5,July,Артилерія та РСЗВ,NaN,NaN,NaN,47.123866,32.621648,2022-7-5,1.656979e+12
53,6,July,Артилерія та РСЗВ,NaN,NaN,NaN,47.198526,32.829574,2022-7-6,1.657066e+12
54,5,July,Артилерія та РСЗВ,NaN,NaN,NaN,47.023995,32.427847,2022-7-5,1.656979e+12
55,5,July,Артилерія та РСЗВ,NaN,NaN,NaN,46.606658,31.565892,2022-7-5,1.656979e+12
56,5,July,Артилерія та РСЗВ,NaN,NaN,NaN,46.659323,31.634256,2022-7-5,1.656979e+12
57,5,July,Артилерія та РСЗВ,NaN,NaN,NaN,47.022603,32.455243,2022-7-5,1.656979e+12
58,6,July,Артилерія та РСЗВ,NaN,NaN,NaN,46.942736,31.552642,2022-7-6,1.657066e+12
59,6,July,Артилерія та РСЗВ,NaN,NaN,NaN,46.870032,32.025611,2022-7-6,1.657066e+12


In [22]:
df.loc[[4]]

,day,month,sheet_name,type,scale,description,la,lo,date,timestamp
4,12,July,Ракети,NaN,NaN,NaN,46.959077,32.011807,2022-7-12,1.657584e+12


In [23]:
del(df['day'])
del(df['month'])

In [24]:
df['timestamp'][1]

1656633600000.0

In [25]:
df.loc[df['sheet_name'].isin(['Загиблі цивільні'])] #show begining of the second dataframe

,sheet_name,type,scale,description,la,lo,date,timestamp
227,Загиблі цивільні,NaN,1.0,"В с. Білозірка пошкоджено будинок, загинула од...",47.090263,32.448873,2022-7-1,1.656634e+12
228,Загиблі цивільні,NaN,1.0,В селищі Луч внаслідок ворожих обстрілів загин...,46.826064,32.219268,2022-7-2,1.656720e+12
229,Загиблі цивільні,NaN,1.0,Внаслідок обстрілів с. Засілля загинула одна л...,47.023995,32.427847,2022-7-5,1.656979e+12
230,Загиблі цивільні,NaN,1.0,Під час обстрілів с. Нечаяне загинула одна люд...,46.942736,31.552642,2022-7-6,1.657066e+12
231,Загиблі цивільні,NaN,2.0,Внаслідок обстрілів с. Українка Галицинівської...,46.824849,32.141298,2022-7-7,1.657152e+12
232,Загиблі цивільні,NaN,1.0,Через ракетний удар по Баштанці загинула жінка.,47.403720,32.438221,2022-7-12,1.657584e+12
233,Загиблі цивільні,NaN,4.0,Через обстріли с. Новоселівка загинуло 4 людини.,46.999938,32.395752,2022-7-13,1.657670e+12
234,Загиблі цивільні,NaN,1.0,Через нічний обстріл Баштанки загинула одна лю...,47.403720,32.438221,2022-7-14,1.657757e+12
235,Загиблі цивільні,NaN,3.0,У с. Шевченкове внаслідок обстрілів загинули т...,46.862150,32.196443,2022-7-16,1.657930e+12
236,Загиблі цивільні,NaN,1.0,Через обстріли у Широківській громаді загинула...,47.123866,32.621648,2022-7-16,1.657930e+12


In [59]:
df.to_csv('july.csv', index=False)

In [62]:
df1 = df.loc[df['description'].isnull()] #show empty rows

In [63]:
df1 #split into dataframes with null description

,sheet_name,scale,description,la,lo,date,timestamp
0,Ракети,NaN,NaN,46.959632,32.006337,2022-07-1,1.656634e+09
1,Ракети,NaN,NaN,46.959077,32.011807,2022-07-1,1.656634e+09
2,Ракети,NaN,NaN,46.959077,32.011807,2022-07-5,1.656979e+09
3,Ракети,NaN,NaN,46.959077,32.011807,2022-07-11,1.657498e+09
4,Ракети,NaN,NaN,46.959077,32.011807,2022-07-12,1.657584e+09
...,...,...,...,...,...,...,...
222,Артилерія та РСЗВ,NaN,NaN,47.286823,33.071037,2022-07-31,1.659226e+09
223,Артилерія та РСЗВ,NaN,NaN,47.366277,33.062555,2022-07-31,1.659226e+09
224,Артилерія та РСЗВ,NaN,NaN,47.312209,32.985710,2022-07-31,1.659226e+09
225,Артилерія та РСЗВ,NaN,NaN,47.365991,32.940963,2022-07-31,1.659226e+09


In [64]:
df2 = df.loc[df['description'].notnull()] #show without rows

In [65]:
df2

,sheet_name,scale,description,la,lo,date,timestamp
227,Загиблі цивільні,1.0,"В селі Білозірка пошкоджено будинок, загинула ...",47.090263,32.448873,2022-07-1,1.656634e+09
228,Загиблі цивільні,1.0,В селищі Луч внаслідок ворожих обстрілів загин...,46.826064,32.219268,2022-07-2,1.656720e+09
229,Загиблі цивільні,1.0,Внаслідок обстрілів села Засілля загинула одна...,47.023995,32.427847,2022-07-5,1.656979e+09
230,Загиблі цивільні,1.0,Під час обстрілів с. Нечаяне одна людина загин...,46.942736,31.552642,2022-07-6,1.657066e+09
231,Загиблі цивільні,2.0,Внаслідок обстрілів села Українка Галицинівськ...,46.824849,32.141298,2022-07-7,1.657152e+09
...,...,...,...,...,...,...,...
415,Цивільна інфраструктура,NaN,У Миколаєві ракети влучили в автосалон та сало...,46.959077,32.011810,2022-07-18,1.658102e+09
416,Цивільна інфраструктура,NaN,Влучання в будівлю магазину та гаражний коопер...,47.041280,32.438662,2022-07-22,1.658448e+09
417,Цивільна інфраструктура,NaN,Ракетний удар по Миколаєву пошкодив яхт-клуб.,46.959077,32.011812,2022-07-28,1.658966e+09
418,Цивільна інфраструктура,NaN,У Миколаєві пошкоджено пункт видачі гуманітарн...,46.959077,32.011812,2022-07-28,1.658966e+09


In [118]:
concatenated = pd.DataFrame() #create empty dataframe

In [120]:
concatenated = pd.concat([concatenated, df]) #concate multiple tables into one

In [27]:
# from here https://notebook.community/captainsafia/nteract/applications/desktop/example-notebooks/pandas-to-geojson
def df_to_geojson(df, properties, lat='la', lon='lo'):
    # create a new python dict to contain our geojson data, using geojson format
    geojson = {'type':'FeatureCollection', 'features':[]}

    # loop through each row in the dataframe and convert each row to geojson format
    for _, row in df.iterrows():
        # create a feature template to fill in
        feature = {'type':'Feature',
                   'properties':{},
                   'geometry':{'type':'Point',
                               'coordinates':[]}}

        # fill in the coordinates
        feature['geometry']['coordinates'] = [row[lon],row[lat]]

        # for each column, get the value and add it as a new feature property
        for prop in properties:
            feature['properties'][prop] = row[prop]
        
        # add this feature (aka, converted dataframe row) to the list of features inside our dict
        geojson['features'].append(feature)
    
    return geojson

In [32]:
cols = ['day', 'month', 'sheet_name']
geojson = df_to_geojson(df1, cols)

In [32]:
cols = ['day', 'month', 'sheet_name']
geojson = df_to_geojson(df1, cols)

In [25]:
with open('july_targets2.geojson', 'w', encoding='utf-8') as f:
    json.dump(geojson, f, ensure_ascii=False)

In [9]:
df_texty = pd.read_csv('../texty_map/Copy of fire_points - data_to_use.csv')

In [17]:
filtered_values = np.where((df_texty['Область'].str.startswith('Миколаїв')))
print(filtered_values)
new_df_texty = df_texty.loc[filtered_values]

(array([   95,   103,   104,   110,   225,   263,   264,   274,   319,
         322,   326,   443,   466,   467,   468,   473,   481,   482,
         487,   491,   537,   538,   639,   679,   680,   743,   744,
         759,   821,   870,   944,  1006,  1325,  1327,  1727,  1754,
        1764,  1765,  1766,  1771,  1772,  1909,  1934,  2013,  2016,
        2061,  2085,  2091,  2106,  2113,  2115,  2164,  2175,  2200,
        2235,  2296,  2360,  2417,  2418,  2431,  2447,  2448,  2465,
        2495,  2498,  2521,  2522,  2523,  2540,  2541,  2542,  2551,
        2588,  2589,  2590,  2591,  2592,  2633,  2634,  2635,  2636,
        2637,  2645,  2685,  2686,  2687,  2688,  2689,  2705,  2706,
        2757,  2759,  2774,  2794,  2837,  2838,  2839,  2868,  2869,
        2870,  2958,  2968,  3013,  3014,  3077,  3087,  3109,  3110,
        3111,  3112,  3113,  3126,  3127,  3148,  3149,  3150,  3151,
        3152,  3153,  3200,  3207,  3221,  3230,  3231,  3262,  3267,
        3275,  3294

In [18]:
new_df.to_csv('texty_Mykolaiv.csv', index=False)


In [10]:
df_shell = pd.read_csv('../xls_datasets/df2.csv')

In [11]:
df_shell["scale"].fillna(1, inplace = True) #replace all null values in a colum

In [12]:
cond = df_shell['sheet_name'].str.endswith('цивільні')
print(df_shell[cond].to_string())

            sheet_name  scale                                                                                                                                                                                                                                                                                              description         la         lo        date      timestamp
0     Загиблі цивільні    1.0                                                                                                                                                                                                                                                 В с. Білозірка пошкоджено будинок, загинула одна людина.  47.090263  32.448873    2022-7-1  1656633600000
1     Загиблі цивільні    1.0                                                                                                                                                                                                                           

In [26]:
filtered_values = np.where((df_shell['sheet_name'].str.endswith('цивільні'))&(df_shell['date'].str.startswith('2022-8')))
print(filtered_values)
new_df = df_shell.loc[filtered_values]

(array([193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205,
       206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218,
       219, 220, 221, 222, 223, 224, 225, 226, 227]),)


In [30]:
new_df.sort_values('la')

,sheet_name,scale,description,la,lo,date,timestamp
207,Поранені цивільні,3.0,В селі Парутине внаслідок обстрілу троє людей ...,46.706472,31.896191,2022-8-4,1659571200000
195,Загиблі цивільні,1.0,Внаслідок обстрілу села Галицинове було вбито ...,46.780528,31.952911,2022-8-7,1659830400000
217,Поранені цивільні,3.0,В результаті влучання касетних боєприпасів РСЗ...,46.780528,31.952911,2022-8-18,1660780800000
205,Поранені цивільні,1.0,Через обстріли с. Галицинове поранено одну людину,46.780528,31.952911,2022-8-3,1659484800000
225,Поранені цивільні,1.0,Ворог здійснював обстріли Шевченківської грома...,46.885644,32.227655,2022-8-28,1661644800000
206,Поранені цивільні,2.0,"Було обстріляно село Зоря, як наслідок поранен...",46.896352,32.288157,2022-8-3,1659484800000
224,Поранені цивільні,1.0,Під ворожими обстрілами були с. Оленівка та с-...,46.935270,32.305433,2022-8-27,1661558400000
193,Загиблі цивільні,1.0,Внаслідок масованого обстрілу Миколаєва одна л...,46.959077,32.011812,2022-8-5,1659657600000
216,Поранені цивільні,1.0,Місто зазнало обстрілів ракетами типу С-300. О...,46.959077,32.011812,2022-8-19,1660867200000
213,Поранені цивільні,1.0,У Миколаєві поранення отримала одна цивільна о...,46.959077,32.011812,2022-8-13,1660348800000


In [14]:
df_shell.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 735 entries, 0 to 734
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sheet_name   735 non-null    object 
 1   scale        735 non-null    float64
 2   description  735 non-null    object 
 3   la           735 non-null    float64
 4   lo           735 non-null    float64
 5   date         735 non-null    object 
 6   timestamp    735 non-null    int64  
dtypes: float64(3), int64(1), object(3)
memory usage: 40.3+ KB


In [28]:
cols = ['sheet_name', 'scale', 'description', 'date', 'timestamp']
geojson = df_to_geojson(df_shell, cols)

In [30]:
with open('../data3_withScale.geojson', 'w', encoding='utf-8') as f:
    json.dump(geojson, f, ensure_ascii=False)